# Importador histórico ChileCompra — versión multitabla experimental

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/MercadoPublico-Inteligencia/blob/main/colab/Importador_Historicos_ChileCompra_Multitabla.ipynb)

[Volver al README](https://github.com/juliopez/MercadoPublico-Inteligencia#readme)

**Destino exclusivamente:** clon de Google Sheets. **No invoca API ChileCompra ni GitHub Actions.**

Esta versión admite una tabla de destino por ejecución: `Procesos`, `Ofertas`, `Resultados` y `OrdenesCompra`. Las hojas `Control`, `ControlV2`, `Indicadores` y el resumen histórico no se cargan: no son tablas de importación independientes.

**Seguridad:** inicia en simulación, exige columnas y claves probatorias mínimas, no inventa proveedores/adjudicaciones, no actualiza registros existentes, genera reporte de calidad y requiere confirmación explícita para escribir. Esta ampliación aún necesita contrastar columnas de archivos reales de ofertas, adjudicaciones y OC.
**Consolidación OC:** `cantidad_lineas` requiere `IDItem` consistente en todas las filas de una orden. `total_lineas_neto` solo se suma si existe un importe neto explícito por línea; `precioNeto` es precio unitario y no se suma directamente. Órdenes con cabeceras o IDItem contradictorios bloquean la simulación. Campos no comprobables se dejan vacíos.

## Operación en una sola sesión
Ejecuta una vez hasta **Simulación y preparación**. Revisa el reporte; después ejecuta exclusivamente **Autorización y carga**, ingresando la frase de confirmación en esa celda. No vuelvas a subir el ZIP ni a ejecutar las celdas de transformación. El archivo preparado se guarda en Drive y puede recuperarse tras reiniciar Colab.

**Moneda:** los campos monetarios se convierten a CLP enteros solamente al autorizar su escritura, conservando un informe de ajustes. El reporte de simulación permanece sin cambios.


In [ ]:
!pip -q install pandas gspread google-api-python-client google-auth
from google.colab import auth, files
from google.auth import default
from googleapiclient.discovery import build
import pandas as pd, gspread, io, zipfile, csv, re, unicodedata, os, json, time, datetime
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds)
drive=build('drive','v3',credentials=creds)
print('Autenticación lista')

## Configuración — una tabla por archivo y ejecución
No cambies `CLON_ID`. `ESCRIBIR=False` significa **sin modificaciones**.

In [ ]:
CLON_ID='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
TABLA='OrdenesCompra'   # Procesos | Ofertas | Resultados | OrdenesCompra
FUENTE='compra_agil'  # licitaciones | compra_agil | ordenes_compra
ESCRIBIR=False
CONFIRMACION=''
LOTE=250
assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
assert TABLA in ('Procesos','Ofertas','Resultados','OrdenesCompra')
assert FUENTE in ('licitaciones','compra_agil','ordenes_compra')
assert (TABLA,FUENTE) not in [('Ofertas','ordenes_compra'),('OrdenesCompra','licitaciones'),('OrdenesCompra','compra_agil')]
libro=gc.open_by_key(CLON_ID)
ws=libro.worksheet(TABLA)

In [ ]:
ESQUEMAS={
'Procesos':['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad'],
'Ofertas':['codigo','proveedor','monto_ofertado_clp','fecha_oferta','fuente','id_cotizacion','seleccionado'],
'Resultados':['codigo','adjudicatario','monto_adjudicado_clp','fecha_resultado','motivo_seleccion','fuente','validado','id_cotizacion_ganadora','monto_oferta_seleccionada_clp','estado_verificacion_oc'],
'OrdenesCompra':['codigo_oc','codigo_proceso','fecha_creacion','fecha_aceptacion','estado','proveedor','monto_total_oc_clp','tipo_oc','cantidad_lineas','total_lineas_neto']}
assert ws.row_values(1)==ESQUEMAS[TABLA], 'Cabeceras incompatibles: abortado'
def norm(v):
 s=unicodedata.normalize('NFKD',str(v))
 return re.sub('[^a-z0-9]','', ''.join(c for c in s if not unicodedata.combining(c)).lower())
def val(x):
 s=str(x).strip()
 return '' if s.lower() in ('nan','null','none','nat','<na>') else s
def cod(x):return val(x).upper()
def fecha(x):
 s=val(x)
 if not s:return ''
 try:
  d=pd.to_datetime(s,errors='coerce',dayfirst=not bool(re.match(r'^\d{4}-',s)))
  return d.strftime('%Y-%m-%d') if pd.notna(d) else ''
 except Exception:return ''
def monto(x):
 s=val(x).replace('$','').replace(' ','')
 if not s:return ''
 if re.fullmatch(r'-?\d{1,3}(\.\d{3})+(,\d+)?',s):s=s.replace('.','').replace(',','.')
 elif ',' in s and '.' not in s:s=s.replace(',','.')
 try:
  from decimal import Decimal
  d=Decimal(s)
  return format(d,'f') if d.is_finite() else ''
 except Exception:return ''
ALIAS={
'codigo':['CodigoExterno','CodigoCotizacion','CodigoProceso','CodigoLicitacion','Codigo'],
'nombre':['Nombre','NombreLicitacion','NombreCotizacion','Titulo'],
'estado':['Estado','EstadoLicitacion','EstadoOC'],
'fecha_publicacion':['FechaPublicacion','FechaCreacion'],
'fecha_cierre':['FechaCierre','FechaFinalizacion'],
'monto_disponible_clp':['MontoEstimado','MontoDisponible','Presupuesto'],
'organismo_comprador':['NombreOrganismo','OrganismoComprador'],
'url_detalle':['Link','UrlDetalle'],
'proveedor':['NombreProveedor','Proveedor','RazonSocial','NombreRazonSocial'],
'monto_ofertado_clp':['MontoOfertado','MontoOferta','MontoCotizacion','Monto'],
'fecha_oferta':['FechaOferta','FechaCotizacion'],
'id_cotizacion':['IdCotizacion','CodigoCotizacionDetalle','IdOferta','CodigoOferta'],
'seleccionado':['Seleccionado','Adjudicado','EsGanador'],
'adjudicatario':['Adjudicatario','NombreAdjudicatario','ProveedorAdjudicado'],
'monto_adjudicado_clp':['MontoAdjudicado','MontoAdjudicacion'],
'fecha_resultado':['FechaAdjudicacion','FechaResultado'],
'motivo_seleccion':['MotivoSeleccion','JustificacionAdjudicacion'],
'id_cotizacion_ganadora':['IdCotizacionGanadora','CodigoCotizacionGanadora'],
'codigo_oc':['CodigoOC','CodigoOrdenCompra','CodigoOrdenDeCompra','Codigo'],
'codigo_proceso':['CodigoLicitacion','CodigoProceso','CodigoExternoProceso'],
'fecha_creacion':['FechaCreacion','FechaEnvio','FechaOC'],
'fecha_aceptacion':['FechaAceptacion','fechaAceptacion'],
'monto_total_oc_clp':['MontoTotalOC_PesosChilenos','MontoTotalOC','MontoTotal','TotalOC'],
'tipo_oc':['TipoOC','TipoOrdenCompra'],
'cantidad_lineas':['CantidadLineas','CantidadItems'],
'total_lineas_neto':['TotalLineasNeto','MontoNeto'],
}
# Claves obligatorias: oferta debe tener id individual; resultado necesita evidencia explícita.
OBLIGATORIOS={
'Procesos':['codigo','nombre'],
'Ofertas':['codigo','proveedor','monto_ofertado_clp','id_cotizacion'],
'Resultados':['codigo','adjudicatario','monto_adjudicado_clp'],
'OrdenesCompra':['codigo_oc','proveedor','monto_total_oc_clp']}
CLAVES={'Procesos':['codigo'],'Ofertas':['id_cotizacion'],'Resultados':['codigo'],'OrdenesCompra':['codigo_oc']}
print('Esquema verificado:',TABLA,ESQUEMAS[TABLA])

## Lectura del ZIP y control de correspondencias

Se detiene si el histórico no incluye los campos probatorios de la tabla. No se usarán datos de una cotización para inventar adjudicaciones. Para Ofertas, debe haber ID de cotización único. Los formatos nuevos requieren revisar el reporte antes de habilitar escritura.

In [ ]:
subidos=files.upload()
assert len(subidos)==1,'Sube un solo ZIP o CSV'
nombre=next(iter(subidos))
assert nombre.lower().endswith(('.zip','.csv'))
def contenidos(nombre,bytes_):
 if nombre.lower().endswith('.csv'):yield nombre,bytes_;return
 with zipfile.ZipFile(io.BytesIO(bytes_)) as z:
  csvs=[x for x in z.namelist() if x.lower().endswith('.csv') and not x.startswith('__MACOSX/')]
  assert csvs,'No hay CSV en el ZIP'
  for n in csvs:yield n,z.read(n)
def detector(raw):
 # Validar TODO el CSV: la muestra inicial puede ser cp1252 y fallar al avanzar.
 # latin-1 es fallback sin errores de decodificación, pero se registra el encoding.
 for enc in ('utf-8-sig','utf-8','cp1252','latin-1'):
  try:
   texto=raw.decode(enc,errors='strict')
  except UnicodeDecodeError:
   continue
  linea=texto.splitlines()[0] if texto.splitlines() else ''
  sep=max([';',',','\t','|'],key=linea.count)
  if linea.count(sep)>0:return enc,sep
 raise ValueError('CSV sin encabezado o delimitador reconocible')
def equivalencias(cols):
 d={norm(x):x for x in cols}
 return {k:next((d[norm(a)] for a in aliases if norm(a) in d),None) for k,aliases in ALIAS.items()}
def transformar(row,m):
 r={k:val(row.get(m.get(k),'') if m.get(k) else '') for k in ESQUEMAS[TABLA]}
 for k in ['codigo','codigo_oc','codigo_proceso']: 
  if k in r:r[k]=cod(r[k])
 for k in r:
  if k.startswith('fecha_'):r[k]=fecha(r[k])
  if k.startswith('monto_') or k=='total_lineas_neto':r[k]=monto(r[k])
 if TABLA=='Procesos':r['modalidad']='Compra Ágil' if FUENTE=='compra_agil' else 'Licitación'
 if TABLA in ('Ofertas','Resultados'):r['fuente']='Histórico ChileCompra '+FUENTE+' '+nombre
 if TABLA=='Resultados':
  r['validado']='Pendiente de verificación de OC'
  r['estado_verificacion_oc']='OC no verificada'
 return r
filas=0;invalidas=0;detalle={};diagnosticos=[]
lineas_por_oc={}; conflictivas=set(); oc_inconsistentes=set(); sin_iditem=set()
for item,raw in contenidos(nombre,subidos[nombre]):
 enc,sep=detector(raw)
 first=True;cont=0
 for chunk in pd.read_csv(io.BytesIO(raw),encoding=enc,sep=sep,dtype=str,keep_default_na=False,chunksize=20000,on_bad_lines='error'):
  if first:
   encabezados={norm(c) for c in chunk.columns}
   if 'montototalocpesoschilenos' in encabezados and TABLA!='OrdenesCompra':
    raise ValueError('Este CSV corresponde a Órdenes de Compra. Configura TABLA="OrdenesCompra" y FUENTE="ordenes_compra".')
   if TABLA=='OrdenesCompra' and 'montototalocpesoschilenos' not in encabezados and not ({'montototaloc','montototal','totaloc'} & encabezados):
    raise ValueError('No se reconoce un monto total de OC. Comprueba que el ZIP sea de Órdenes de Compra.')
   m=equivalencias(chunk.columns)
   if TABLA=='OrdenesCompra':
    nombres={norm(x):x for x in chunk.columns}
    id_item_col=nombres.get('iditem')
    total_linea_col=next((nombres[k] for k in ('totallineaneto','montonetolinea','totalnetolinea') if k in nombres),None)
    diagnosticos.append({'detalle_OC':{'id_item':id_item_col,'total_neto_explicito':total_linea_col,'nota':'PrecioNeto no es total de línea; no se multiplica ni suma automáticamente.'}})
   missing=[k for k in OBLIGATORIOS[TABLA] if not m.get(k)]
   assert not missing, f'{item}: faltan columnas esenciales {missing}. Encontradas: {list(chunk.columns)}'
   diagnosticos.append({'archivo':item,'codificacion':enc,'delimitador':sep,'columnas':m})
   first=False
  for row in chunk.to_dict('records'):
   filas+=1;cont+=1
   r=transformar(row,m)
   if any(not r.get(k) for k in OBLIGATORIOS[TABLA]):invalidas+=1;continue
   clave=tuple(r[k] for k in CLAVES[TABLA])
   if TABLA=='OrdenesCompra':
    oc=r['codigo_oc']; item_id=val(row.get(id_item_col,'')) if id_item_col else ''
    if item_id:
     key_item=(oc,item_id)
     registro=(val(row.get('cantidad','')),val(row.get('precioNeto','')),val(row.get(total_linea_col,'')) if total_linea_col else '')
     bucket=lineas_por_oc.setdefault(oc,{})
     if key_item in bucket and bucket[key_item]!=registro:conflictivas.add(oc)
     else:bucket[key_item]=registro
    else:sin_iditem.add(oc)

   prev=detalle.get(clave)
   if prev is not None:
    if TABLA=='OrdenesCompra':
     for campo in ('monto_total_oc_clp','proveedor','codigo_proceso'):
      if prev.get(campo) and r.get(campo) and prev[campo]!=r[campo]:oc_inconsistentes.add(r['codigo_oc'])
    # Ofertas duplicadas con mismo ID y valores diferentes => detener.
    if TABLA=='Ofertas' and (prev['codigo']!=r['codigo'] or prev['proveedor']!=r['proveedor'] or prev['monto_ofertado_clp']!=r['monto_ofertado_clp']):
     raise ValueError('Conflicto de datos para oferta '+repr(clave))
    continue
   detalle[clave]=r
 print(item,cont,'filas')
if TABLA=='OrdenesCompra':
 for (oc,),r in detalle.items():
  if oc in conflictivas or oc in oc_inconsistentes:
   r['cantidad_lineas']='';r['total_lineas_neto']='';continue
  items=lineas_por_oc.get(oc,{})
  # Solo contar IDItem distintos cuando todas las filas tienen identificador.
  r['cantidad_lineas']=str(len(items)) if items and oc not in sin_iditem else ''
  # Nunca sumar montos OC repetidos ni precios unitarios.
  # Sumar únicamente montos NETOS de línea explícitos y completos, si existen.
  if total_linea_col and items and oc not in sin_iditem:
   from decimal import Decimal, InvalidOperation
   valores=[monto(x[2]) for x in items.values()]
   try:r['total_lineas_neto']=str(sum(Decimal(x) for x in valores)) if all(x!='' for x in valores) else ''
   except InvalidOperation:r['total_lineas_neto']=''
  else:r['total_lineas_neto']=''
 assert not oc_inconsistentes, f'OC con cabeceras contradictorias: {len(oc_inconsistentes)}. No cargar antes de revisar.'
 assert not conflictivas, f'OC con IDItem conflictivos: {len(conflictivas)}. No cargar antes de revisar.'
assert detalle,'Sin registros válidos'
print('Preparación completada')

In [ ]:
# Se leen claves de la hoja real, aun si el clon está vacío.
col_indices=[ESQUEMAS[TABLA].index(c)+1 for c in CLAVES[TABLA]]
cols_existentes=[ws.col_values(i)[1:] for i in col_indices]
n=max(map(len,cols_existentes),default=0)
existentes=[]
for i in range(n):
 key=tuple(cod(col[i]) if i<len(col) else '' for col in cols_existentes)
 if all(key):existentes.append(key)
assert len(existentes)==len(set(existentes)),'Hay claves duplicadas en el destino'
previas=set(existentes)
nuevos=[r for k,r in detalle.items() if k not in previas]
reporte={'archivo':nombre,'fuente':FUENTE,'tabla':TABLA,'filas_originales':filas,'filas_invalidas':invalidas,
         'claves_unicas_validas':len(detalle),'duplicados_en_origen':filas-invalidas-len(detalle),
         'ya_existentes':len(detalle)-len(nuevos),'nuevos':len(nuevos),
         'total_destino_inicial':len(previas),'total_destino_esperado':len(previas)+len(nuevos),
         'mapeos':diagnosticos}
if TABLA=='OrdenesCompra':
 reporte['oc_sin_id_item_completo']=len(sin_iditem)
 reporte['oc_con_cantidad_lineas']=sum(bool(r['cantidad_lineas']) for r in detalle.values())
 reporte['oc_con_total_lineas_neto']=sum(bool(r['total_lineas_neto']) for r in detalle.values())
 reporte['politica_montos']='Nunca sumar monto total OC repetido ni precio neto unitario; campo neto vacío sin monto neto explícito por línea.'
print(json.dumps(reporte,indent=2,ensure_ascii=False))
display(pd.DataFrame(nuevos[:5],columns=ESQUEMAS[TABLA]))
stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('/content/reporte_'+stamp+'.json','w') as f:json.dump(reporte,f,ensure_ascii=False,indent=2)
pd.DataFrame(nuevos,columns=ESQUEMAS[TABLA]).to_csv('/content/nuevos_'+stamp+'.csv',index=False,encoding='utf-8-sig')
# Persistir datos preparados SIN volver a consultar los archivos históricos.
# Drive permite recuperar una sesión interrumpida con el ID mostrado.
from googleapiclient.http import MediaFileUpload
import hashlib
preparado_local='/content/preparado_'+stamp+'.json'
paquete={'clon_id':CLON_ID,'tabla':TABLA,'fuente':FUENTE,'columnas':ESQUEMAS[TABLA],
         'claves':CLAVES[TABLA],'reporte':reporte,'registros':nuevos}
with open(preparado_local,'w',encoding='utf-8') as fp:json.dump(paquete,fp,ensure_ascii=False)
digest=hashlib.sha256(open(preparado_local,'rb').read()).hexdigest()
archivo_drive=drive.files().create(body={'name':'Observatorio_preparado_'+TABLA+'_'+stamp+'.json','description':'Datos históricos preparados para carga al CLON exclusivamente'},
 media_body=MediaFileUpload(preparado_local,mimetype='application/json',resumable=True),
 fields='id,name,webViewLink').execute()
PREPARADO_DRIVE_ID=archivo_drive['id']
print('PREPARADO_DRIVE_ID:',PREPARADO_DRIVE_ID)
print('SHA256:',digest)
print('Preparación guardada en Drive; puedes autorizar la carga SIN volver a procesar el ZIP.')


## Autorización y carga (ejecutar solo esta celda después de revisar el informe)

Si los recuentos son correctos, cambia `CONFIRMAR_AQUI` a `AUTORIZO CARGA EN CLON` y ejecuta **solo esta celda**. No reinicies el entorno ni repitas las celdas superiores. Se crea una copia de respaldo de la hoja clonada antes de insertar. En caso de fallo, los registros preparados quedan guardados en Drive; puedes reanudar sin el ZIP desde la celda de recuperación al final.

**Importante:** Google Sheets tiene límites de tiempo y de tamaño; no ejecutes otra sesión de carga en paralelo.
**Política monetaria:** para la exportación al clon los montos en CLP se normalizan a **pesos enteros** mediante `Decimal` y redondeo `ROUND_HALF_UP`. Se muestra un resumen de cuántos valores cambiaron y ejemplos *antes* de escribir. Esto no reconstruye los valores textuales originales del ZIP: si se requiere precisión literal, habrá que validar el CSV original. La celda utiliza el paquete ya preparado, no vuelve a cargar el ZIP.


In [ ]:
CONFIRMAR_AQUI = ''   # escribir exactamente: AUTORIZO CARGA EN CLON
# Política monetaria del destino: CLP enteros, redondeo HALF_UP.
# Esta corrección usa el paquete ya preparado, sin releer el ZIP.
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
CAMPOS_CLP=[k for k in ESQUEMAS[TABLA] if k.startswith('monto_') or k=='total_lineas_neto']
def normalizar_clp_preparado(registros):
 cambios={k:0 for k in CAMPOS_CLP}
 muestras=[]
 salida=[]
 for r in registros:
  row=dict(r)
  for campo in CAMPOS_CLP:
   raw=str(row.get(campo,'')).strip()
   if not raw:continue
   try:
    dec=Decimal(raw)
    assert dec.is_finite()
    entero=str(dec.quantize(Decimal('1'),rounding=ROUND_HALF_UP))
   except (InvalidOperation, AssertionError) as e:
    raise ValueError(f'Monto no interpretable en {campo}, codigo {row.get("codigo_oc",row.get("codigo",""))}: {raw}') from e
   if entero!=raw:
    cambios[campo]+=1
    if len(muestras)<8:muestras.append({'codigo':row.get('codigo_oc',row.get('codigo','')),'campo':campo,'antes':raw,'despues':entero})
   row[campo]=entero
  salida.append(row)
 return salida,cambios,muestras
REGISTROS_NORMALIZADOS,AJUSTES_CLP,MUESTRAS_CLP=normalizar_clp_preparado(paquete['registros'])
print('Normalización CLP sobre datos preparados:',AJUSTES_CLP)
print('Ejemplos:',MUESTRAS_CLP)
print('Registros preparados:',len(REGISTROS_NORMALIZADOS))
if CONFIRMAR_AQUI!='AUTORIZO CARGA EN CLON':
 print('Carga desactivada. El reporte y los datos preparados se conservan.')
else:
 assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 assert paquete['clon_id']==CLON_ID and paquete['tabla']==TABLA
 assert paquete['columnas']==ESQUEMAS[TABLA] and paquete['claves']==CLAVES[TABLA]
 assert ws.row_values(1)==ESQUEMAS[TABLA]
 copia=drive.files().copy(fileId=CLON_ID,body={'name':'Respaldo clon antes de carga '+TABLA+' '+stamp},fields='id,webViewLink').execute()
 assert copia['id']!=CLON_ID
 print('RESPALDO:',copia.get('webViewLink',copia['id']))
 # Cargar claves existentes UNA vez; actualización en memoria tras cada lote confirmado.
 columnas=[ws.col_values(j)[1:] for j in col_indices]
 count=max(map(len,columnas),default=0)
 current_rows=[tuple(cod(c[x]) if x<len(c) else '' for c in columnas) for x in range(count)]
 assert all(all(k) for k in current_rows),'Claves vacías: abortado'
 assert len(current_rows)==len(set(current_rows)),'Duplicados existentes: abortado'
 current=set(current_rows)
 pendientes_de_escribir=[r for r in REGISTROS_NORMALIZADOS if tuple(cod(r[k]) for k in CLAVES[TABLA]) not in current]
 print('Por insertar ahora:',len(pendientes_de_escribir))
 escritos=0
 from gspread.exceptions import APIError
 for i in range(0,len(pendientes_de_escribir),LOTE):
  bloque=pendientes_de_escribir[i:i+LOTE]
  # Cada fila del destino se agrega al final de una tabla sin huecos en columna A.
  pos=len(current)+2
  fin=pos+len(bloque)-1
  rango=f'A{pos}:{chr(64+len(ESQUEMAS[TABLA]))}{fin}'
  valores=[[str(r.get(c,'')) for c in ESQUEMAS[TABLA]] for r in bloque]
  try:
   ws.update(range_name=rango,values=valores,value_input_option='RAW')
  except APIError:
   print('Error de escritura: NO repetir el lote sin verificar. Reejecuta esta celda tras revisar; deduplicará lo ya guardado.')
   raise
  filas_verificadas=ws.get(rango)
  claves_verificadas={tuple(cod(v[ESQUEMAS[TABLA].index(k)]) for k in CLAVES[TABLA]) for v in filas_verificadas}
  claves_bloque={tuple(cod(r[k]) for k in CLAVES[TABLA]) for r in bloque}
  assert claves_verificadas==claves_bloque,'Lote no verificado: detener escritura'
  current.update(claves_bloque)
  escritos+=len(bloque)
  print('Insertados:',escritos,'de',len(pendientes_de_escribir))
  time.sleep(1)
 print('CARGA TERMINADA:',escritos,'nuevos; total:',len(current))


## Recuperación opcional sin ZIP
Úsala **solo si se perdió la sesión de Colab**: pega el ID de Drive mostrado durante la preparación, ejecuta esta celda y después vuelve a la celda «Autorización y carga». Mantén la misma tabla/fuente en Configuración. No procesa nuevamente el histórico.

In [ ]:
RECUPERAR_ID = ''  # pegar PREPARADO_DRIVE_ID si hubo reinicio del entorno
if RECUPERAR_ID:
 from googleapiclient.http import MediaIoBaseDownload
 b=io.BytesIO()
 solicitud=drive.files().get_media(fileId=RECUPERAR_ID)
 descarga=MediaIoBaseDownload(b,solicitud)
 terminado=False
 while not terminado:_,terminado=descarga.next_chunk()
 paquete=json.loads(b.getvalue().decode('utf-8'))
 assert paquete['clon_id']=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 assert paquete['tabla']==TABLA and paquete['fuente']==FUENTE
 assert paquete['columnas']==ESQUEMAS[TABLA] and paquete['claves']==CLAVES[TABLA]
 nuevos=paquete['registros']
 col_indices=[ESQUEMAS[TABLA].index(c)+1 for c in CLAVES[TABLA]]
 stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
 print('Recuperados',len(nuevos),'registros preparados. Revisa reporte y ejecuta SOLO la celda de autorización.')
 print(json.dumps(paquete['reporte'],ensure_ascii=False,indent=2))
else:
 print('No es necesaria la recuperación si mantienes la sesión actual.')